In [1]:
!pip install scipy

# ============================================================
# Q3 ATTENTION BENCHMARK — FOCUS SHIFTING ACROSS SUBTASKS
# ------------------------------------------------------------
# Question:
# Can the model shift focus between sub-tasks in a complex, multi-part prompt
# without losing track?
#
# Core design:
# - Keep article content fixed.
# - No distractors. Keep article text fixed across conditions.
# - Change ONLY the subtask burden and order.
#
# Conditions:
# 1) single_focus
#    Main task only: predict EPU.
#
# 2) multi_focus_epu_first
#    Same context, but solve multiple subtasks with EPU first.
#
# 3) multi_focus_epu_last
#    Same context, same subtasks, but EPU comes last.
#
# Why this is stronger than the old combined notebook:
# - It isolates focus shifting rather than length.
# - It separates Q3 from Q2/Q4 cleanly.
# - It exports direct scientific proof:
#   accuracy, paired drops, flip rates, McNemar p-values, bootstrap CIs,
#   subtask completion/correctness, and hardest harmed rows.
# ============================================================

import hashlib
import json
import math
import re
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import binomtest
from tqdm.auto import tqdm

import kaggle_benchmarks as kbench

# ---------------------------
# CONFIG
# ---------------------------
INPUT_CSV = "/kaggle/input/datasets/bigdataanalysis1/epu-800/EPU_benchmark_800_balanced.csv"

SEED = 42
N_ROWS = 600            # pilot size; raise to 600 when stable
MAX_CHARS = 5000
BOOT_N = 5000

MODEL_NAMES = [
    # "google/gemini-2.5-flash",
    "google/gemini-2.5-pro",
    # "anthropic/claude-sonnet-4-6@default",
    # "deepseek-ai/deepseek-v3.2",
    # "qwen/qwen3-235b-a22b-instruct-2507"
]

OUT_DIR = Path("/kaggle/working/epu_attention_q3_focus_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MISSING_SENTINEL = -1

# ---------------------------
# LOAD + CLEAN
# ---------------------------
df = pd.read_csv(INPUT_CSV)

required_cols = [
    "article_key",
    "content",
    "gold_epu",
    "gold_who",
    "gold_actions",
    "gold_effects",
    "mention_foreign",
    "mainly_foreign",
    "disagreement_flag",
    "newspaper",
    "year",
    "content_len",
]
for c in required_cols:
    if c not in df.columns:
        df[c] = np.nan

mask = (
    df["content"].notna()
    & (df["content"].astype(str).str.len() > 300)
    & df["gold_epu"].notna()
)

clean = df.loc[mask].copy()
clean["article_key"] = clean["article_key"].astype(str)
missing_key_mask = clean["article_key"].isin(["nan", "None", ""])
if missing_key_mask.any():
    clean.loc[missing_key_mask, "article_key"] = [
        f"generated_key_{i}" for i in range(missing_key_mask.sum())
    ]

clean["gold_epu"] = clean["gold_epu"].astype(int)
clean["gold_mention_foreign"] = (clean["mention_foreign"].fillna(0) >= 0.5).astype(int)
clean["gold_mainly_foreign"] = (clean["mainly_foreign"].fillna(0) >= 0.5).astype(int)
clean["gold_who"] = clean["gold_who"].fillna(MISSING_SENTINEL).astype(int)
clean["gold_actions"] = clean["gold_actions"].fillna(MISSING_SENTINEL).astype(int)
clean["gold_effects"] = clean["gold_effects"].fillna(MISSING_SENTINEL).astype(int)
clean["disagreement_flag"] = clean["disagreement_flag"].fillna(0).astype(int)
clean = clean.drop_duplicates(subset=["article_key"]).reset_index(drop=True)

print(f"Loaded rows: {len(df):,}")
print(f"Clean usable rows: {len(clean):,}")
print(clean["gold_epu"].value_counts().sort_index())
print(clean["disagreement_flag"].value_counts().sort_index())

# ---------------------------
# BALANCED SAMPLE
# ---------------------------
def balanced_sample(data: pd.DataFrame, n: int, seed: int = 42) -> pd.DataFrame:
    pos = data[data["gold_epu"] == 1].copy()
    neg = data[data["gold_epu"] == 0].copy()

    if len(pos) == 0 or len(neg) == 0:
        return data.sample(min(n, len(data)), random_state=seed).reset_index(drop=True)

    n_pos = min(len(pos), n // 2)
    n_neg = min(len(neg), n - n_pos)

    pos_s = pos.sample(n=n_pos, random_state=seed)
    neg_s = neg.sample(n=n_neg, random_state=seed + 1)

    return (
        pd.concat([pos_s, neg_s], axis=0)
        .sample(frac=1, random_state=seed + 2)
        .reset_index(drop=True)
    )

pilot = balanced_sample(clean, N_ROWS, SEED).copy()

# ---------------------------
# DISTRACTOR CONTEXT
# Keep context constant across all Q3 conditions.
# ---------------------------

def smart_truncate(text: str, max_chars: int = 5000) -> str:
    text = str(text)
    if len(text) <= max_chars:
        return text
    head = max_chars // 2
    tail = max_chars - head - 32
    return text[:head] + "\n\n[...TRUNCATED...]\n\n" + text[-tail:]

pilot["content_for_eval"] = pilot["content"].astype(str).apply(lambda x: smart_truncate(x, MAX_CHARS))
pilot["medium_pre"] = ""
pilot["medium_post"] = ""

pilot_path = OUT_DIR / "pilot_rows_q3.csv"
pilot.to_csv(pilot_path, index=False)

# ---------------------------
# HELPERS
# ---------------------------
AUX_FIELDS = ["who", "actions", "effects", "mention_foreign", "mainly_foreign"]

def maybe_int(x):
    if x is None:
        return None
    try:
        if pd.isna(x):
            return None
    except Exception:
        pass
    try:
        return int(x)
    except Exception:
        return None

def binary_match(pred, gold, missing_sentinel=MISSING_SENTINEL):
    gold_i = maybe_int(gold)
    pred_i = maybe_int(pred)
    if gold_i is None or gold_i == missing_sentinel:
        return None
    if pred_i is None:
        return 0.0
    return 1.0 if pred_i == gold_i else 0.0

def safe_float(x, default=np.nan):
    try:
        return float(x)
    except Exception:
        return default

def default_out():
    return {
        "epu": None,
        "who": None,
        "actions": None,
        "effects": None,
        "mention_foreign": None,
        "mainly_foreign": None,
        "confidence": 0.0,
        "focus_excerpt": "",
        "raw_text": "",
        "json_valid": 0,
    }

def parse_llm_json(raw_text):
    out = default_out()
    out["raw_text"] = "" if raw_text is None else str(raw_text)[:4000]

    if raw_text is None:
        return out

    text = str(raw_text).strip()
    text = text.replace("```json", "").replace("```", "").strip()

    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        text = match.group(0)

    try:
        obj = json.loads(text)
        if not isinstance(obj, dict):
            return out
    except Exception:
        return out

    out["json_valid"] = 1

    for k in out:
        if k in obj and k not in ["raw_text", "json_valid"]:
            out[k] = obj[k]

    for k in ["epu", "who", "actions", "effects", "mention_foreign", "mainly_foreign"]:
        out[k] = maybe_int(out[k])

    try:
        out["confidence"] = float(out["confidence"])
    except Exception:
        out["confidence"] = 0.0

    out["confidence"] = max(0.0, min(1.0, out["confidence"]))
    out["focus_excerpt"] = "" if out["focus_excerpt"] is None else str(out["focus_excerpt"])[:250]
    return out

def model_json_call(llm, prompt: str):
    try:
        raw = llm.prompt(prompt)
        return parse_llm_json(raw)
    except Exception as e:
        out = default_out()
        out["raw_text"] = f"ERROR: {str(e)[:4000]}"
        return out

def excerpt_in_target(excerpt: str, article_text: str) -> int:
    excerpt = "" if excerpt is None else str(excerpt).strip()
    article_text = "" if article_text is None else str(article_text)
    if len(excerpt) < 12:
        return 0
    return int(excerpt in article_text)

# ---------------------------
# PROMPTS
# ---------------------------

def build_single_prompt(article_text: str, prefix: str = "", suffix: str = "") -> str:
    return f"""You are evaluating the ARTICLE below.

Task:
Decide whether the ARTICLE expresses ECONOMIC POLICY UNCERTAINTY (EPU).

Labeling rule:
- EPU = 1 only if the ARTICLE discusses uncertainty about government policy,
  policy decisions, possible policy actions, political outcomes affecting policy,
  or the economic effects of policy.
- EPU = 0 for general economics, elections, markets, war, uncertainty, or foreign events
  unless the uncertainty is specifically tied to economic policy.

IMPORTANT:
- Base your answer only on the ARTICLE below.
- Return ONLY valid JSON.
- No markdown. No explanation. No code fence.

Return exactly these keys:
{{
  "epu": 0 or 1,
  "confidence": number between 0 and 1,
  "focus_excerpt": "short quote from the article only"
}}

ARTICLE START
{article_text}
ARTICLE END""".strip()

def build_multi_prompt(article_text: str, prefix: str = "", suffix: str = "", order: str = "epu_first") -> str:
    if order == "epu_first":
        task_block = """Perform ALL subtasks below in order:
1. Decide whether the ARTICLE expresses ECONOMIC POLICY UNCERTAINTY (EPU).
2. Code whether the uncertainty is about who.
3. Code whether the uncertainty is about actions.
4. Code whether the uncertainty is about effects.
5. Code mention_foreign.
6. Code mainly_foreign.
7. Provide a short supporting quote from the ARTICLE only.

"""
    elif order == "epu_last":
        task_block = """Perform ALL subtasks below in order:
1. Code whether the uncertainty is about who.
2. Code whether the uncertainty is about actions.
3. Code whether the uncertainty is about effects.
4. Code mention_foreign.
5. Code mainly_foreign.
6. Provide a short supporting quote from the ARTICLE only.
7. Only after finishing the above, decide whether the ARTICLE expresses ECONOMIC POLICY UNCERTAINTY (EPU).

"""
    else:
        raise ValueError(f"Unknown order: {order}")

    return f"""You are evaluating the ARTICLE below.

{task_block}Labeling rule:
- EPU = 1 only if the ARTICLE discusses uncertainty about government policy,
  policy decisions, possible policy actions, political outcomes affecting policy,
  or the economic effects of policy.
- EPU = 0 for general economics, elections, markets, war, uncertainty, or foreign events
  unless the uncertainty is specifically tied to economic policy.

IMPORTANT:
- Base your answers only on the ARTICLE below.
- Return ONLY valid JSON.
- No markdown. No explanation. No code fence.

Return exactly these keys:
{{
  "epu": 0 or 1,
  "who": 0 or 1 or -1,
  "actions": 0 or 1 or -1,
  "effects": 0 or 1 or -1,
  "mention_foreign": 0 or 1,
  "mainly_foreign": 0 or 1,
  "confidence": number between 0 and 1,
  "focus_excerpt": "short quote from the article only"
}}

ARTICLE START
{article_text}
ARTICLE END""".strip()

# ---------------------------
# STATISTICS
# ---------------------------
def wilson_ci(successes: int, n: int, z: float = 1.959963984540054):
    if n <= 0:
        return (np.nan, np.nan)
    phat = successes / n
    denom = 1.0 + (z**2) / n
    center = (phat + (z**2) / (2 * n)) / denom
    margin = (z / denom) * math.sqrt((phat * (1 - phat) / n) + (z**2) / (4 * n**2))
    return (max(0.0, center - margin), min(1.0, center + margin))

def paired_bootstrap_diff_ci(a_values, b_values, n_boot=5000, seed=42, alpha=0.05):
    a = np.asarray(a_values, dtype=float)
    b = np.asarray(b_values, dtype=float)
    mask = ~(np.isnan(a) | np.isnan(b))
    a = a[mask]
    b = b[mask]
    n = len(a)
    if n == 0:
        return (np.nan, np.nan)
    diffs = a - b
    rng = np.random.default_rng(seed)
    boot = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        idx = rng.integers(0, n, size=n)
        boot[i] = diffs[idx].mean()
    return (float(np.quantile(boot, alpha / 2)), float(np.quantile(boot, 1 - alpha / 2)))

def exact_mcnemar(a_correct, b_correct):
    x = np.asarray(a_correct, dtype=float)
    y = np.asarray(b_correct, dtype=float)
    mask = ~(np.isnan(x) | np.isnan(y))
    x = x[mask].astype(int)
    y = y[mask].astype(int)
    b = int(np.sum((x == 1) & (y == 0)))
    c = int(np.sum((x == 0) & (y == 1)))
    n = b + c
    if n == 0:
        return {
            "b_hurt": b,
            "c_helped": c,
            "discordant_n": n,
            "mcnemar_p": 1.0,
            "hurt_help_ratio": np.nan,
        }
    pval = binomtest(min(b, c), n=n, p=0.5, alternative="two-sided").pvalue
    ratio = np.inf if c == 0 and b > 0 else (b / c if c > 0 else np.nan)
    return {
        "b_hurt": b,
        "c_helped": c,
        "discordant_n": n,
        "mcnemar_p": float(pval),
        "hurt_help_ratio": ratio,
    }

def classify_shift(drop_pp, flip_rate, pval):
    if pd.isna(drop_pp) or pd.isna(flip_rate):
        return "undetermined"
    if drop_pp >= 0.10 and flip_rate >= 0.15 and pval < 0.05:
        return "strong focus-shift failure"
    if drop_pp >= 0.03 and flip_rate >= 0.05 and pval < 0.05:
        return "moderate focus-shift failure"
    if drop_pp < 0.03 and flip_rate < 0.05:
        return "little focus-shift failure"
    return "mixed / borderline"

def summarize_pair(a_correct, b_correct, a_pred, b_pred, compare_name):
    a = np.asarray(a_correct, dtype=float)
    b = np.asarray(b_correct, dtype=float)
    pred_a = np.asarray(a_pred, dtype=float)
    pred_b = np.asarray(b_pred, dtype=float)

    mask = ~(np.isnan(a) | np.isnan(b))
    a = a[mask]
    b = b[mask]
    pred_a = pred_a[mask]
    pred_b = pred_b[mask]

    n = len(a)
    acc_a = float(np.mean(a)) if n else np.nan
    acc_b = float(np.mean(b)) if n else np.nan
    drop_pp = acc_a - acc_b if n else np.nan
    drop_lo, drop_hi = paired_bootstrap_diff_ci(a, b, n_boot=BOOT_N, seed=SEED)

    flip_mask = ~(np.isnan(pred_a) | np.isnan(pred_b))
    flip_n = int(np.sum(flip_mask))
    flip_k = int(np.sum(pred_a[flip_mask] != pred_b[flip_mask])) if flip_n else 0
    flip_rate = flip_k / flip_n if flip_n else np.nan
    flip_lo, flip_hi = wilson_ci(flip_k, flip_n)

    hurt_k = int(np.sum((a == 1) & (b == 0))) if n else 0
    help_k = int(np.sum((a == 0) & (b == 1))) if n else 0
    mc = exact_mcnemar(a, b)

    return {
        "comparison": compare_name,
        "n_rows": n,
        "acc_a": acc_a,
        "acc_b": acc_b,
        "drop_pp": drop_pp,
        "drop_ci_low": drop_lo,
        "drop_ci_high": drop_hi,
        "flip_rate": flip_rate,
        "flip_rate_ci_low": flip_lo,
        "flip_rate_ci_high": flip_hi,
        "hurt_rate": hurt_k / n if n else np.nan,
        "help_rate": help_k / n if n else np.nan,
        **mc,
        "shift_level": classify_shift(drop_pp, flip_rate, mc["mcnemar_p"]),
    }

# ---------------------------
# RUN MODELS
# ---------------------------
records = []
model_handles = {name: kbench.llms[name] for name in MODEL_NAMES}

for model_name in MODEL_NAMES:
    llm = model_handles[model_name]
    print(f"\nRunning {model_name} ...")

    for row in tqdm(pilot.to_dict(orient="records"), total=len(pilot), desc=model_name):
        content = row["content_for_eval"]
        pre = row["medium_pre"]
        post = row["medium_post"]

        out_single = model_json_call(llm, build_single_prompt(content, pre, post))
        out_first = model_json_call(llm, build_multi_prompt(content, pre, post, order="epu_first"))
        out_last = model_json_call(llm, build_multi_prompt(content, pre, post, order="epu_last"))

        rec = {
            "llm_name": model_name,
            "article_key": row["article_key"],
            "gold_epu": row["gold_epu"],
            "gold_who": row["gold_who"],
            "gold_actions": row["gold_actions"],
            "gold_effects": row["gold_effects"],
            "gold_mention_foreign": row["gold_mention_foreign"],
            "gold_mainly_foreign": row["gold_mainly_foreign"],
            "disagreement_flag": row["disagreement_flag"],
            "newspaper": row.get("newspaper", ""),
            "year": row.get("year", np.nan),
            "content_len": row.get("content_len", np.nan),
            "content_for_eval": content,
            "medium_pre": pre,
            "medium_post": post,
        }

        outputs = {
            "single_focus": out_single,
            "multi_focus_epu_first": out_first,
            "multi_focus_epu_last": out_last,
        }

        for cond, out in outputs.items():
            rec[f"{cond}_epu_pred"] = maybe_int(out["epu"])
            rec[f"{cond}_epu_correct"] = binary_match(out["epu"], row["gold_epu"])
            rec[f"{cond}_confidence"] = safe_float(out["confidence"])
            rec[f"{cond}_focus_excerpt"] = out["focus_excerpt"]
            rec[f"{cond}_excerpt_in_target"] = excerpt_in_target(out["focus_excerpt"], content)
            rec[f"{cond}_json_valid"] = int(out["json_valid"])
            rec[f"{cond}_raw_json"] = out["raw_text"]

            aux_scores = []
            for field in AUX_FIELDS:
                gold = row[f"gold_{field}"]
                pred = maybe_int(out[field])
                correct = binary_match(pred, gold)
                rec[f"{cond}_{field}_pred"] = pred
                rec[f"{cond}_{field}_correct"] = correct
                rec[f"{cond}_{field}_present"] = int(pred is not None)
                if correct is not None:
                    aux_scores.append(float(correct))
            rec[f"{cond}_aux_mean"] = np.nan if len(aux_scores) == 0 else float(np.mean(aux_scores))
            rec[f"{cond}_task_completion_count"] = int(sum(rec[f"{cond}_{field}_present"] for field in AUX_FIELDS))

        rec["single_to_first_hurt"] = int(rec["single_focus_epu_correct"] == 1.0 and rec["multi_focus_epu_first_epu_correct"] == 0.0) if not pd.isna(rec["single_focus_epu_correct"]) and not pd.isna(rec["multi_focus_epu_first_epu_correct"]) else 0
        rec["single_to_last_hurt"] = int(rec["single_focus_epu_correct"] == 1.0 and rec["multi_focus_epu_last_epu_correct"] == 0.0) if not pd.isna(rec["single_focus_epu_correct"]) and not pd.isna(rec["multi_focus_epu_last_epu_correct"]) else 0

        records.append(rec)

wide_df = pd.DataFrame(records)
wide_path = OUT_DIR / "q3_results_wide.csv"
wide_df.to_csv(wide_path, index=False)

# ---------------------------
# LONG VERSION
# ---------------------------
long_records = []
conditions = ["single_focus", "multi_focus_epu_first", "multi_focus_epu_last"]

for row in wide_df.to_dict(orient="records"):
    base = {
        "llm_name": row["llm_name"],
        "article_key": row["article_key"],
        "gold_epu": row["gold_epu"],
        "gold_who": row["gold_who"],
        "gold_actions": row["gold_actions"],
        "gold_effects": row["gold_effects"],
        "gold_mention_foreign": row["gold_mention_foreign"],
        "gold_mainly_foreign": row["gold_mainly_foreign"],
        "disagreement_flag": row["disagreement_flag"],
        "newspaper": row["newspaper"],
        "year": row["year"],
        "content_len": row["content_len"],
    }
    for cond in conditions:
        rec = dict(base)
        rec["condition"] = cond
        rec["epu_pred"] = row[f"{cond}_epu_pred"]
        rec["epu_correct"] = row[f"{cond}_epu_correct"]
        rec["confidence"] = row[f"{cond}_confidence"]
        rec["focus_excerpt"] = row[f"{cond}_focus_excerpt"]
        rec["excerpt_in_target"] = row[f"{cond}_excerpt_in_target"]
        rec["json_valid"] = row[f"{cond}_json_valid"]
        rec["aux_mean"] = row[f"{cond}_aux_mean"]
        rec["task_completion_count"] = row[f"{cond}_task_completion_count"]
        for field in AUX_FIELDS:
            rec[f"{field}_pred"] = row[f"{cond}_{field}_pred"]
            rec[f"{field}_correct"] = row[f"{cond}_{field}_correct"]
            rec[f"{field}_present"] = row[f"{cond}_{field}_present"]
        long_records.append(rec)

long_df = pd.DataFrame(long_records)
long_path = OUT_DIR / "q3_results_long.csv"
long_df.to_csv(long_path, index=False)

# ---------------------------
# SUMMARY TABLES
# ---------------------------
summary_rows = []
for model_name, g in wide_df.groupby("llm_name", sort=False):
    rec = {"llm_name": model_name, "n_rows": len(g)}

    for cond in conditions:
        vec = g[f"{cond}_epu_correct"].astype(float).to_numpy()
        n = int(np.sum(~np.isnan(vec)))
        k = int(np.nansum(vec)) if n > 0 else 0
        mean_acc = float(np.nanmean(vec)) if n > 0 else np.nan
        lo, hi = wilson_ci(k, n)

        rec[f"{cond}_acc"] = mean_acc
        rec[f"{cond}_acc_ci_low"] = lo
        rec[f"{cond}_acc_ci_high"] = hi
        rec[f"{cond}_json_valid_rate"] = g[f"{cond}_json_valid"].mean()
        rec[f"{cond}_excerpt_in_target_rate"] = g[f"{cond}_excerpt_in_target"].mean()
        rec[f"{cond}_confidence_mean"] = g[f"{cond}_confidence"].mean()
        rec[f"{cond}_aux_mean"] = g[f"{cond}_aux_mean"].mean()
        rec[f"{cond}_completion_mean"] = g[f"{cond}_task_completion_count"].mean()

    comp_single_first = summarize_pair(
        g["single_focus_epu_correct"],
        g["multi_focus_epu_first_epu_correct"],
        g["single_focus_epu_pred"],
        g["multi_focus_epu_first_epu_pred"],
        "single_focus -> multi_focus_epu_first",
    )
    comp_single_last = summarize_pair(
        g["single_focus_epu_correct"],
        g["multi_focus_epu_last_epu_correct"],
        g["single_focus_epu_pred"],
        g["multi_focus_epu_last_epu_pred"],
        "single_focus -> multi_focus_epu_last",
    )
    comp_first_last = summarize_pair(
        g["multi_focus_epu_first_epu_correct"],
        g["multi_focus_epu_last_epu_correct"],
        g["multi_focus_epu_first_epu_pred"],
        g["multi_focus_epu_last_epu_pred"],
        "multi_focus_epu_first -> multi_focus_epu_last",
    )

    rec["single_to_first_drop_pp"] = comp_single_first["drop_pp"]
    rec["single_to_first_mcnemar_p"] = comp_single_first["mcnemar_p"]
    rec["single_to_first_flip_rate"] = comp_single_first["flip_rate"]

    rec["single_to_last_drop_pp"] = comp_single_last["drop_pp"]
    rec["single_to_last_drop_ci_low"] = comp_single_last["drop_ci_low"]
    rec["single_to_last_drop_ci_high"] = comp_single_last["drop_ci_high"]
    rec["single_to_last_mcnemar_p"] = comp_single_last["mcnemar_p"]
    rec["single_to_last_flip_rate"] = comp_single_last["flip_rate"]
    rec["single_to_last_shift_level"] = comp_single_last["shift_level"]

    rec["first_to_last_drop_pp"] = comp_first_last["drop_pp"]
    rec["first_to_last_mcnemar_p"] = comp_first_last["mcnemar_p"]
    rec["first_to_last_flip_rate"] = comp_first_last["flip_rate"]

    summary_rows.append(rec)

summary_overall = pd.DataFrame(summary_rows).sort_values("single_to_last_drop_pp", ascending=False)
summary_overall_path = OUT_DIR / "q3_summary_overall.csv"
summary_overall.to_csv(summary_overall_path, index=False)

pairwise_rows = []
for model_name, g in wide_df.groupby("llm_name", sort=False):
    pairwise_rows.append(dict({"llm_name": model_name}, **summarize_pair(
        g["single_focus_epu_correct"],
        g["multi_focus_epu_first_epu_correct"],
        g["single_focus_epu_pred"],
        g["multi_focus_epu_first_epu_pred"],
        "single_focus -> multi_focus_epu_first",
    )))
    pairwise_rows.append(dict({"llm_name": model_name}, **summarize_pair(
        g["single_focus_epu_correct"],
        g["multi_focus_epu_last_epu_correct"],
        g["single_focus_epu_pred"],
        g["multi_focus_epu_last_epu_pred"],
        "single_focus -> multi_focus_epu_last",
    )))
    pairwise_rows.append(dict({"llm_name": model_name}, **summarize_pair(
        g["multi_focus_epu_first_epu_correct"],
        g["multi_focus_epu_last_epu_correct"],
        g["multi_focus_epu_first_epu_pred"],
        g["multi_focus_epu_last_epu_pred"],
        "multi_focus_epu_first -> multi_focus_epu_last",
    )))

pairwise_df = pd.DataFrame(pairwise_rows)
pairwise_path = OUT_DIR / "q3_pairwise_proof.csv"
pairwise_df.to_csv(pairwise_path, index=False)

# ---------------------------
# BY DISAGREEMENT / BY GOLD LABEL
# ---------------------------
def summarize_wide_group(g: pd.DataFrame):
    rec = {"n_rows": len(g)}
    for cond in conditions:
        rec[f"{cond}_acc"] = g[f"{cond}_epu_correct"].mean()
        rec[f"{cond}_json_valid_rate"] = g[f"{cond}_json_valid"].mean()
        rec[f"{cond}_excerpt_in_target_rate"] = g[f"{cond}_excerpt_in_target"].mean()
        rec[f"{cond}_aux_mean"] = g[f"{cond}_aux_mean"].mean()
    comp = summarize_pair(
        g["single_focus_epu_correct"],
        g["multi_focus_epu_last_epu_correct"],
        g["single_focus_epu_pred"],
        g["multi_focus_epu_last_epu_pred"],
        "single_focus -> multi_focus_epu_last",
    )
    rec["single_to_last_drop_pp"] = comp["drop_pp"]
    rec["single_to_last_mcnemar_p"] = comp["mcnemar_p"]
    rec["single_to_last_flip_rate"] = comp["flip_rate"]
    rec["single_to_last_shift_level"] = comp["shift_level"]
    return rec

by_disagreement_rows = []
for (model_name, flag), g in wide_df.groupby(["llm_name", "disagreement_flag"], sort=False):
    by_disagreement_rows.append(
        dict({"llm_name": model_name, "disagreement_flag": int(flag)}, **summarize_wide_group(g))
    )

by_disagreement_df = pd.DataFrame(by_disagreement_rows)
by_disagreement_path = OUT_DIR / "q3_summary_by_disagreement.csv"
by_disagreement_df.to_csv(by_disagreement_path, index=False)

by_gold_rows = []
for (model_name, gold_epu), g in wide_df.groupby(["llm_name", "gold_epu"], sort=False):
    by_gold_rows.append(
        dict({"llm_name": model_name, "gold_epu": int(gold_epu)}, **summarize_wide_group(g))
    )

by_gold_df = pd.DataFrame(by_gold_rows)
by_gold_path = OUT_DIR / "q3_summary_by_gold_epu.csv"
by_gold_df.to_csv(by_gold_path, index=False)

# ---------------------------
# SUBTASK BEHAVIOR TABLE
# ---------------------------
field_order_map = {
    "multi_focus_epu_first": {
        "epu": 1, "who": 2, "actions": 3, "effects": 4, "mention_foreign": 5, "mainly_foreign": 6
    },
    "multi_focus_epu_last": {
        "who": 1, "actions": 2, "effects": 3, "mention_foreign": 4, "mainly_foreign": 5, "epu": 6
    },
}

subtask_rows = []
for model_name, g in wide_df.groupby("llm_name", sort=False):
    for cond in ["multi_focus_epu_first", "multi_focus_epu_last"]:
        # EPU itself
        epu_vec = g[f"{cond}_epu_correct"].astype(float).to_numpy()
        n = int(np.sum(~np.isnan(epu_vec)))
        k = int(np.nansum(epu_vec)) if n > 0 else 0
        lo, hi = wilson_ci(k, n)
        subtask_rows.append({
            "llm_name": model_name,
            "condition": cond,
            "field": "epu",
            "prompt_order_index": field_order_map[cond]["epu"],
            "completion_rate": g[f"{cond}_json_valid"].mean(),
            "correctness_mean": float(np.nanmean(epu_vec)) if n > 0 else np.nan,
            "correctness_ci_low": lo,
            "correctness_ci_high": hi,
        })

        for field in AUX_FIELDS:
            present = g[f"{cond}_{field}_present"].mean()
            vec = g[f"{cond}_{field}_correct"].astype(float).to_numpy()
            n = int(np.sum(~np.isnan(vec)))
            k = int(np.nansum(vec)) if n > 0 else 0
            lo, hi = wilson_ci(k, n)
            subtask_rows.append({
                "llm_name": model_name,
                "condition": cond,
                "field": field,
                "prompt_order_index": field_order_map[cond][field],
                "completion_rate": present,
                "correctness_mean": float(np.nanmean(vec)) if n > 0 else np.nan,
                "correctness_ci_low": lo,
                "correctness_ci_high": hi,
            })

subtask_df = pd.DataFrame(subtask_rows)
subtask_path = OUT_DIR / "q3_subtask_table.csv"
subtask_df.to_csv(subtask_path, index=False)

# ---------------------------
# HARDEST / MOST HARMED ROWS
# ---------------------------
harmed_rows = (
    wide_df.groupby(["article_key", "gold_epu", "disagreement_flag"], as_index=False)
    .agg(
        harmed_models_single_to_last=("single_to_last_hurt", "sum"),
        harmed_models_single_to_first=("single_to_first_hurt", "sum"),
        mean_single_acc=("single_focus_epu_correct", "mean"),
        mean_first_acc=("multi_focus_epu_first_epu_correct", "mean"),
        mean_last_acc=("multi_focus_epu_last_epu_correct", "mean"),
        content_for_eval=("content_for_eval", "first"),
        medium_pre=("medium_pre", "first"),
        medium_post=("medium_post", "first"),
    )
    .sort_values(["harmed_models_single_to_last", "mean_last_acc"], ascending=[False, True])
)
harmed_path = OUT_DIR / "q3_most_harmed_rows.csv"
harmed_rows.to_csv(harmed_path, index=False)

# ---------------------------
# MANIFEST
# ---------------------------
manifest = pd.DataFrame({
    "file_name": [
        wide_path.name,
        long_path.name,
        summary_overall_path.name,
        pairwise_path.name,
        by_disagreement_path.name,
        by_gold_path.name,
        subtask_path.name,
        harmed_path.name,
        pilot_path.name,
    ],
    "description": [
        "Row-level wide table with one row per model x article and predictions for single_focus, multi_focus_epu_first, and multi_focus_epu_last.",
        "Long-format table with one row per model x article x condition.",
        "Main Q3 summary table with condition accuracies and the strongest pairwise focus-shift effect.",
        "Paired scientific proof for single->first, single->last, and first->last comparisons.",
        "Q3 metrics split by disagreement_flag.",
        "Q3 metrics split by gold_epu.",
        "Per-subtask completion and correctness table to show which tasks get lost as prompt burden changes.",
        "Rows most harmed when EPU must be answered last rather than alone.",
        "The exact pilot rows used for Q3.",
    ]
})
manifest_path = OUT_DIR / "q3_manifest.csv"
manifest.to_csv(manifest_path, index=False)

print("\nSaved files:")
for p in [
    wide_path, long_path, summary_overall_path, pairwise_path, by_disagreement_path,
    by_gold_path, subtask_path, harmed_path, pilot_path, manifest_path
]:
    print(p)

print("\nQ3 overall summary:")
display(summary_overall)

print("\nQ3 pairwise proof:")
display(pairwise_df)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/35.3 MB ? eta -:--:--

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 35.1/35.3 MB 215.4 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 111.7 MB/s  0:00:00


Loaded rows: 800
Clean usable rows: 800
gold_epu
0    400
1    400
Name: count, dtype: int64
disagreement_flag
0    585
1    215
Name: count, dtype: int64



Running google/gemini-2.5-pro ...


google/gemini-2.5-pro:   0%|          | 0/600 [00:00<?, ?it/s]


Saved files:
/kaggle/working/epu_attention_q3_focus_outputs/q3_results_wide.csv
/kaggle/working/epu_attention_q3_focus_outputs/q3_results_long.csv
/kaggle/working/epu_attention_q3_focus_outputs/q3_summary_overall.csv
/kaggle/working/epu_attention_q3_focus_outputs/q3_pairwise_proof.csv
/kaggle/working/epu_attention_q3_focus_outputs/q3_summary_by_disagreement.csv
/kaggle/working/epu_attention_q3_focus_outputs/q3_summary_by_gold_epu.csv
/kaggle/working/epu_attention_q3_focus_outputs/q3_subtask_table.csv
/kaggle/working/epu_attention_q3_focus_outputs/q3_most_harmed_rows.csv
/kaggle/working/epu_attention_q3_focus_outputs/pilot_rows_q3.csv
/kaggle/working/epu_attention_q3_focus_outputs/q3_manifest.csv

Q3 overall summary:


,llm_name,n_rows,single_focus_acc,single_focus_acc_ci_low,single_focus_acc_ci_high,single_focus_json_valid_rate,single_focus_excerpt_in_target_rate,single_focus_confidence_mean,single_focus_aux_mean,single_focus_completion_mean,...,single_to_first_flip_rate,single_to_last_drop_pp,single_to_last_drop_ci_low,single_to_last_drop_ci_high,single_to_last_mcnemar_p,single_to_last_flip_rate,single_to_last_shift_level,first_to_last_drop_pp,first_to_last_mcnemar_p,first_to_last_flip_rate
0,google/gemini-2.5-pro,600,0.45,0.410637,0.49,0.683333,0.576667,0.66325,0.0,0.0,...,0.088235,-0.005,-0.028333,0.016667,0.765992,0.102941,mixed / borderline,-0.011667,0.296206,0.071253



Q3 pairwise proof:


,llm_name,comparison,n_rows,acc_a,acc_b,drop_pp,drop_ci_low,drop_ci_high,flip_rate,flip_rate_ci_low,flip_rate_ci_high,hurt_rate,help_rate,b_hurt,c_helped,discordant_n,mcnemar_p,hurt_help_ratio,shift_level
0,google/gemini-2.5-pro,single_focus -> multi_focus_epu_first,600,0.450000,0.443333,0.006667,-0.013333,0.026667,0.088235,0.064415,0.119737,0.035000,0.028333,21,17,38,0.627103,1.235294,mixed / borderline
1,google/gemini-2.5-pro,single_focus -> multi_focus_epu_last,600,0.450000,0.455000,-0.005000,-0.028333,0.016667,0.102941,0.077063,0.136226,0.035000,0.040000,21,24,45,0.765992,0.875000,mixed / borderline
2,google/gemini-2.5-pro,multi_focus_epu_first -> multi_focus_epu_last,600,0.443333,0.455000,-0.011667,-0.030000,0.006667,0.071253,0.050066,0.100458,0.021667,0.033333,13,20,33,0.296206,0.650000,mixed / borderline
